# Heart Disease Prediction — Decision Tree Classifier
### Exploratory Data Analysis & Model Building

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import joblib
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/heart.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nTarget distribution:\n', df['target'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['target'].value_counts().plot(kind='bar', ax=axes[0], color=['#3498db','#e67e22'], edgecolor='black')
axes[0].set_title('Target Class Distribution')
axes[0].set_xticklabels(['No Disease (0)', 'Disease (1)'], rotation=0)
df['target'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                  colors=['#3498db','#e67e22'], startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Target Distribution (Pie)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='target', kde=True, bins=20, palette={0:'#3498db', 1:'#e67e22'})
plt.title('Age Distribution by Heart Disease Status')
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
cont_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, len(cont_features), figsize=(18, 5))
for i, feat in enumerate(cont_features):
    sns.boxplot(data=df, x='target', y=feat, ax=axes[i], palette={0:'#3498db', 1:'#e67e22'})
    axes[i].set_title(feat)
plt.suptitle('Continuous Features by Target', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
cat_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for i, feat in enumerate(cat_features):
    row, col = divmod(i, 4)
    sns.countplot(data=df, x=feat, hue='target', ax=axes[row][col], palette={0:'#3498db', 1:'#e67e22'})
    axes[row][col].set_title(feat)
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 4. Finding Optimal Tree Depth

In [ ]:
depths = range(1, 21)
train_acc, test_acc = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_sc, y_train)
    train_acc.append(accuracy_score(y_train, dt.predict(X_train_sc)))
    test_acc.append(accuracy_score(y_test,  dt.predict(X_test_sc)))
best_depth = depths[np.argmax(test_acc)]
print(f'Best Depth: {best_depth}  |  Best Test Accuracy: {max(test_acc):.4f}')
plt.figure(figsize=(12, 5))
plt.plot(depths, train_acc, marker='o', label='Train Accuracy', color='#3498db')
plt.plot(depths, test_acc,  marker='s', label='Test Accuracy',  color='#e67e22')
plt.axvline(best_depth, linestyle='--', color='gray', label=f'Best Depth={best_depth}')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree Accuracy vs Max Depth')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Train Final Model & Evaluate

In [ ]:
dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt.fit(X_train_sc, y_train)
y_pred = dt.predict(X_test_sc)
y_prob = dt.predict_proba(X_test_sc)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, y_pred, target_names=['No Disease','Disease']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease','Disease'], yticklabels=['No Disease','Disease'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#e67e22', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1], 'gray', linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#e67e22', edgecolor='black', linewidth=0.5)
plt.title('Feature Importances — Decision Tree Classifier')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(22, 8))
plot_tree(dt, feature_names=X.columns.tolist(), class_names=['No Disease','Disease'],
          filled=True, rounded=True, fontsize=9, max_depth=3)
plt.title('Decision Tree Structure')
plt.tight_layout()
plt.show()

In [ ]:
cv_scores = cross_val_score(dt, scaler.transform(X), y, cv=10, scoring='accuracy')
print(f'10-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 6. Save Model & Scaler

In [ ]:
joblib.dump(dt,     '../models/dt_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('dt_model.pkl and scaler.pkl saved to models/')